In [1]:
library(auk)
library(tidyverse)

library(magrittr)
library(httr)
library(data.table)
library(readr)

library(lubridate)
library(hms)

readRenviron("~/.Renviron")
R.home()

auk 0.8.0 is designed for EBD files downloaded after 2024-10-29. 
No EBD data directory set, see ?auk_set_ebd_path to set EBD_PATH 
eBird taxonomy version:  2024

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘magrittr’


The following object is masked from ‘package:purrr’:

    set_names


The following object is masked from ‘package:tidyr’:

    extract



Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter

[1] "/usr/lib/R"

In [2]:
PROJECT_DIR = getwd()

ebd_dir = file.path(PROJECT_DIR, "source_data", "ebd_relMar-2026")
ebd_filename = "ebd_relMar-2026.txt"

sed_dir = file.path(PROJECT_DIR, "source_data", "ebd_sampling_relMar-2026")
sed_filename = "ebd_sampling_relMar-2026.txt"

In [ ]:
auk_version()
auk_ebd_version(file.path(ebd_dir, ebd_filename))

$auk_version
[1] "auk 0.8.0"

$ebd_version
[1] "2024-10-29"

$taxonomy_version
[1] 2024

<span style="color:deepskyblue">Citation:  
*eBird Basic Dataset. Version: EBD_relSep-2025. Cornell Lab of Ornithology, Ithaca, New York. Sep 2025.*</span>

## Data extraction

In [ ]:
# print all columns
readLines(file.path(ebd_dir, ebd_filename), n = 1) |> strsplit("\t") |> unlist()

[1] "GLOBAL UNIQUE IDENTIFIER"   "LAST EDITED DATE"          
 [3] "TAXONOMIC ORDER"            "CATEGORY"                  
 [5] "TAXON CONCEPT ID"           "COMMON NAME"               
 [7] "SCIENTIFIC NAME"            "SUBSPECIES COMMON NAME"    
 [9] "SUBSPECIES SCIENTIFIC NAME" "EXOTIC CODE"               
[11] "OBSERVATION COUNT"          "BREEDING CODE"             
[13] "BREEDING CATEGORY"          "BEHAVIOR CODE"             
[15] "AGE/SEX"                    "COUNTRY"                   
[17] "COUNTRY CODE"               "STATE"                     
[19] "STATE CODE"                 "COUNTY"                    
[21] "COUNTY CODE"                "IBA CODE"                  
[23] "BCR CODE"                   "USFWS CODE"                
[25] "ATLAS BLOCK"                "LOCALITY"                  
[27] "LOCALITY ID"                "LOCALITY TYPE"             
[29] "LATITUDE"                   "LONGITUDE"                 
[31] "OBSERVATION DATE"           "TIME OBSERVATIONS STARTED" 
[33] "OBSERVER ID"                "OBSERVER ORCID ID"         
[35] "SAMPLING EVENT IDENTIFIER"  "OBSERVATION TYPE"          
[37] "PROTOCOL NAME"              "PROTOCOL CODE"             
[39] "PROJECT NAMES"              "PROJECT IDENTIFIERS"       
[41] "DURATION MINUTES"           "EFFORT DISTANCE KM"        
[43] "EFFORT AREA HA"             "NUMBER OBSERVERS"          
[45] "ALL SPECIES REPORTED"       "GROUP IDENTIFIER"          
[47] "HAS MEDIA"                  "APPROVED"                  
[49] "REVIEWED"                   "REASON"                    
[51] "CHECKLIST COMMENTS"         "SPECIES COMMENTS"

#### <span style="color: green">Combining all raptors into one pull, larger geo range, 2010-2026 only</span>

In [ ]:
file.desc <- "all_raptors_20260821"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_names = c(
    "American Goshawk",
    "American Kestrel",
    "Bald Eagle",
    "Barred Owl",
    "Broad-winged Hawk",
    "Burrowing Owl",
    "Cooper's Hawk",
    "Crested Caracara",
    "Ferruginous Hawk",
    "Flammulated Owl",
    "Golden Eagle",
    "Great Horned Owl",
    "Harris's Hawk",
    "Long-eared Owl",
    "Merlin",
    "Mississippi Kite",
    "Northern Harrier",
    "Osprey",
    "Peregrine Falcon",
    "Red-shouldered Hawk",
    "Red-tailed Hawk",
    "Sharp-shinned Hawk",
    "Short-eared Owl",
    "Swainson's Hawk",
    "Swallow-tailed Kite",
    "White-tailed Kite"
)
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_names) %>%
    auk_bbox(bbox = c(longitude_min = -140, latitude_min = -30, longitude_max = -40, latitude_max = 65)) %>%
    auk_date(c("*-07-01", "*-12-31")) %>%
    auk_year(2010:2026) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_names):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/eBird_raptor_migration_vis/output/auk/ebd_all_raptors_20260821.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/eBird_raptor_migration_vis/output/auk/sed_all_raptors_20260821.txt 

Filters 
  Species: 26 species
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -140 - -40; Lat -30 - 65
  Years: 17 years
  Date: *-07-01 - *-12-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

In [ ]:
species_names = c(
    "American Goshawk",
    "American Kestrel",
    "Bald Eagle",
    "Barred Owl",
    "Broad-winged Hawk",
    "Burrowing Owl",
    "Cooper's Hawk",
    "Crested Caracara",
    "Ferruginous Hawk",
    "Flammulated Owl",
    "Golden Eagle",
    "Great Horned Owl",
    "Harris's Hawk",
    "Long-eared Owl",
    "Merlin",
    "Mississippi Kite",
    "Northern Harrier",
    "Osprey",
    "Peregrine Falcon",
    "Red-shouldered Hawk",
    "Red-tailed Hawk",
    "Sharp-shinned Hawk",
    "Short-eared Owl",
    "Swainson's Hawk",
    "Swallow-tailed Kite",
    "White-tailed Kite"
)
file.desc <- "all_raptors_20260821"
f_full_ebd <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))

prefix <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_"))
species_files <- auk_split(f_full_ebd, species = species_names, prefix = prefix, overwrite = TRUE)

#### Split each file into multiple parts to avoid memory issues with the zero-filling or whatever was having an issue below when input-file size was too large.

In [ ]:
# Run "loop" once w/ sampling data included so it gets split, but redundant to keep it included for every species.
# Will write out the same three split SED files every time (and they're the slowest part)

prefix <- "all_raptors_20260821"
species.names <- c(
 "Accipiter_striatus"
)

for (species.name in species.names) {
    file.desc <- paste0(prefix, "_", species.name)
    f_in_ebd <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
    f_in_sed <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, ".txt"))

    # 1   #######################
    f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_1.txt"))
    f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, "_1.txt"))
    auk_ebd(f_in_ebd
          , file_sampling = f_in_sed     # Have to split SED once for all species, so only include it in first-species run
            ) %>%
        auk_date(c("*-07-01", "*-08-31")) %>%
        auk_filter(file = f_out_ebd_only,
                   file_sampling = f_out_sed_only,      # Have to split SED once for all species, so only include it in first-species run
                   keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                           "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                           "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                           "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                           "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
                   overwrite = TRUE
                   )

    # 2   #######################
    f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_2.txt"))
    f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, "_2.txt"))
    auk_ebd(f_in_ebd
          , file_sampling = f_in_sed     # Have to split SED once for all species, so only include it in first-species run
            ) %>%
        auk_date(c("*-09-01", "*-10-31")) %>%
        auk_filter(file = f_out_ebd_only,
                   file_sampling = f_out_sed_only,      # Have to split SED once for all species, so only include it in first-species run
                   keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                            "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                            "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                            "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                            "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
                   overwrite = TRUE
                   )

    # 3   #######################
    f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_3.txt"))
    f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, "_3.txt"))
    auk_ebd(f_in_ebd
          , file_sampling = f_in_sed     # Have to split SED once for all species, so only included this in first-species run
            ) %>%
        auk_date(c("*-11-01", "*-12-31")) %>%
        auk_filter(file = f_out_ebd_only,
                   file_sampling = f_out_sed_only,      # Have to split SED once for all species, so only included this in first-species run
                   keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                            "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                            "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                            "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                            "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
                   overwrite = TRUE
                   )
}

Warning message in auk_filter.auk_ebd(., file = f_out_ebd_only, file_sampling = f_out_sed_only, :
“Sampling event data file provided, but filters have not been  set to only return complete checklists. Complete checklists  are required for zero-filling. You may want to use  auk_complete(), or manually filter out incomplete checklists.”
Warning message in auk_filter.auk_ebd(., file = f_out_ebd_only, file_sampling = f_out_sed_only, :
“Sampling event data file provided, but filters have not been  set to only return complete checklists. Complete checklists  are required for zero-filling. You may want to use  auk_complete(), or manually filter out incomplete checklists.”
Warning message in auk_filter.auk_ebd(., file = f_out_ebd_only, file_sampling = f_out_sed_only, :
“Sampling event data file provided, but filters have not been  set to only return complete checklists. Complete checklists  are required for zero-filling. You may want to use  auk_complete(), or manually filter out incomplete ch

In [ ]:
prefix <- "all_raptors_20260821"

species.names <- c(
#  "Accipiter_striatus",        # Sharp-shinned Hawk      # run separately above w/ SED included
 "Aquila_chrysaetos",         # Golden Eagle
 "Asio_flammeus",             # Short-eared Owl
 "Asio_otus",                 # Long-eared Owl
 "Astur_atricapillus",        # Northern Goshawk
 "Astur_cooperii",            # Cooper's Hawk
 "Athene_cunicularia",        # Burrowing Owl
 "Bubo_virginianus",          # Great Horned Owl
 "Buteo_jamaicensis",         # Red-tailed Hawk
 "Buteo_lineatus",            # Red-shouldered Hawk
 "Buteo_platypterus",         # Broad-winged Hawk
 "Buteo_regalis",             # Ferruginous Hawk
 "Buteo_swainsoni",           # Swainson's Hawk
 "Caracara_plancus",          # Crested Caracara
 "Circus_hudsonius",          # Northern Harrier
 "Elanoides_forficatus",      # STK
 "Elanus_leucurus",           # White-tailed Kite
 "Falco_columbarius",         # Merlin
 "Falco_peregrinus",          # Peregrine Falcon
 "Falco_sparverius",          # American Kestrel
 "Haliaeetus_leucocephalus",  # Bald Eagle
 "Ictinia_mississippiensis",  # MSK
 "Pandion_haliaetus",         # Osprey
 "Parabuteo_unicinctus",      # Harris's Hawk
 "Psiloscops_flammeolus",     # Flammulated Owl
 "Strix_varia",               # Barred Owl
 )

for (species.name in species.names) {
    file.desc <- paste0(prefix, "_", species.name)
    f_in_ebd <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
    f_in_sed <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, ".txt"))

    # 1   #######################
    f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_1.txt"))
    f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, "_1.txt"))
    auk_ebd(f_in_ebd
          # , file_sampling = f_in_sed     # Have to split SED once for all species, so only included this in first-species run
            ) %>%
        auk_date(c("*-07-01", "*-08-31")) %>%
        auk_filter(file = f_out_ebd_only,
                  #  file_sampling = f_out_sed_only,      # Have to split SED once for all species, so only included this in first-species run
                   keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                           "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                           "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                           "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                           "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
                   overwrite = TRUE
                   )

    # 2   #######################
    f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_2.txt"))
    f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, "_2.txt"))
    auk_ebd(f_in_ebd
          # , file_sampling = f_in_sed     # Have to split SED once for all species, so only included this in first-species run
            ) %>%
        auk_date(c("*-09-01", "*-10-31")) %>%
        auk_filter(file = f_out_ebd_only,
                  #  file_sampling = f_out_sed_only,      # Have to split SED once for all species, so only included this in first-species run
                   keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                            "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                            "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                            "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                            "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
                   overwrite = TRUE
                   )

    # 3   #######################
    f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_3.txt"))
    f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", prefix, "_3.txt"))
    auk_ebd(f_in_ebd
          # , file_sampling = f_in_sed     # Have to split SED once for all species, so only included this in first-species run
            ) %>%
        auk_date(c("*-11-01", "*-12-31")) %>%
        auk_filter(file = f_out_ebd_only,
                  #  file_sampling = f_out_sed_only,      # Have to split SED once for all species, so only included this in first-species run
                   keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                            "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                            "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                            "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                            "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
                  #  overwrite = TRUE
                   )
}

### <span style="color:grey">ARC</span>

#### <span style="color: orange">Swallow-tailed Kite, Fall migration, expanded</span>

In [19]:
file.desc <- "swallow_tailed_kite_compl_exp_range_20260729"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Swallow-tailed Kite"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -108, latitude_min = -15, longitude_max = -50, latitude_max = 40)) %>%
    # auk_date(c("*-08-01", "*-10-31")) %>%
    auk_date(c("*-07-01", "*-11-30")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_swallow_tailed_kite_compl_exp_range_20260729.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_swallow_tailed_kite_compl_exp_range_20260729.txt 

Filters 
  Species: Elanoides forficatus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -108 - -50; Lat -15 - 40
  Years: all
  Date: *-07-01 - *-11-30
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Swallow-tailed Kite, all-time, expanded geo range</span>

In [22]:
file.desc <- "swallow_tailed_kite_compl_alltime_exp_geo_range_20260730"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Swallow-tailed Kite"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -120, latitude_min = -30, longitude_max = -40, latitude_max = 70)) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_swallow_tailed_kite_compl_alltime_exp_geo_range_20260730.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_swallow_tailed_kite_compl_alltime_exp_geo_range_20260730.txt 

Filters 
  Species: Elanoides forficatus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -120 - -40; Lat -30 - 70
  Years: all
  Date: all
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Broad-winged Hawk, Fall migration, expanded</span>

In [ ]:
file.desc <- "broad_winged_hawk_compl_exp_range_20260729"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Broad-winged Hawk"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -108, latitude_min = -15, longitude_max = -50, latitude_max = 40)) %>%
    # auk_date(c("*-08-01", "*-10-31")) %>%
    auk_date(c("*-07-01", "*-11-30")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_broad_winged_hawk_compl_20260726.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_broad_winged_hawk_compl_20260726.txt 

Filters 
  Species: Buteo platypterus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -108 - -50; Lat -15 - 40
  Years: all
  Date: *-07-01 - *-11-30
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Broad-winged Hawk, all-time, expanded geo range</span>

In [23]:
file.desc <- "broad_winged_hawk_compl_alltime_exp_geo_range_20260730"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Broad-winged Hawk"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -120, latitude_min = -30, longitude_max = -40, latitude_max = 70)) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_broad_winged_hawk_compl_alltime_exp_geo_range_20260730.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_broad_winged_hawk_compl_alltime_exp_geo_range_20260730.txt 

Filters 
  Species: Buteo platypterus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -120 - -40; Lat -30 - 70
  Years: all
  Date: all
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Mississippi Kite, Fall migration</span>

In [13]:
file.desc <- "mississippi_kite_compl_20260726"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Mississippi Kite"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -108, latitude_min = -15, longitude_max = -50, latitude_max = 40)) %>%
    auk_date(c("*-08-01", "*-10-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_mississippi_kite_compl_20260726.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_mississippi_kite_compl_20260726.txt 

Filters 
  Species: Ictinia mississippiensis
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -108 - -50; Lat -15 - 40
  Years: all
  Date: *-08-01 - *-10-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Eastern Whip-poor-will, all time</span>

In [4]:
file.desc <- "eastern_wpw_20260726"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Eastern Whip-poor-will"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_country("United States") %>%
    auk_state(c("US-SC", "US-GA", "US-NC", "US-TN", "US-FL")) %>% 
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_eastern_wpw_20260726.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_eastern_wpw_20260726.txt 

Filters 
  Species: Antrostomus vociferus
  Countries: all
  States: US-FL, US-GA, US-NC, US-SC, US-TN
  Counties: all
  BCRs: all
  Bounding box: full extent
  Years: all
  Date: all
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Purple Martin, Fall migration</span>

In [14]:
file.desc <- "purple_martin_compl_20260726"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Purple Martin"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -108, latitude_min = -25, longitude_max = -50, latitude_max = 50)) %>%
    auk_date(c("*-08-01", "*-10-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_purple_martin_compl_20260726.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_purple_martin_compl_20260726.txt 

Filters 
  Species: Progne subis
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -108 - -50; Lat -25 - 50
  Years: all
  Date: *-08-01 - *-10-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Chimney Swift, Fall migration</span>

In [15]:
file.desc <- "chimney_swift_compl_20260726"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Chimney Swift"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -108, latitude_min = -25, longitude_max = -50, latitude_max = 50)) %>%
    auk_date(c("*-08-01", "*-10-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_chimney_swift_compl_20260726.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_chimney_swift_compl_20260726.txt 

Filters 
  Species: Chaetura pelagica
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -108 - -50; Lat -25 - 50
  Years: all
  Date: *-08-01 - *-10-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Peregrine Falcon, Fall migration</span>

In [10]:
file.desc <- "peregrine_falcon_compl_20260727"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Peregrine Falcon"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-08-01", "*-10-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_peregrine_falcon_compl_20260727.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_peregrine_falcon_compl_20260727.txt 

Filters 
  Species: Falco peregrinus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-08-01 - *-10-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Swainson's Hawk, Fall migration</span>

In [ ]:
file.desc <- "swainsons_hawk_compl_20260727"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Swainson's Hawk"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-08-01", "*-10-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_swainsons_hawk_compl_20260727.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_swainsons_hawk_compl_20260727.txt 

Filters 
  Species: Buteo swainsoni
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-08-01 - *-10-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Merlin, Fall migration, expanded geo range</span>

In [25]:
file.desc <- "merlin_compl_20260801"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Merlin"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -120, latitude_min = -30, longitude_max = -40, latitude_max = 70)) %>%
    auk_date(c("*-08-01", "*-12-01")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_merlin_compl_20260801.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_merlin_compl_20260801.txt 

Filters 
  Species: Falco columbarius
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -120 - -40; Lat -30 - 70
  Years: all
  Date: *-08-01 - *-12-01
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Sharp-shinned Hawk, Fall migration, expanded geo range</span>

In [26]:
file.desc <- "ss_hawk_compl_20260801"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Sharp-shinned Hawk"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -120, latitude_min = -30, longitude_max = -40, latitude_max = 70)) %>%
    auk_date(c("*-08-01", "*-12-01")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_ss_hawk_compl_20260801.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_ss_hawk_compl_20260801.txt 

Filters 
  Species: Accipiter striatus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -120 - -40; Lat -30 - 70
  Years: all
  Date: *-08-01 - *-12-01
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Osprey, Fall migration</span>

In [28]:
file.desc <- "osprey_compl_20260801"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Osprey"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-08-01", "*-12-01")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_osprey_compl_20260801.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_osprey_compl_20260801.txt 

Filters 
  Species: Pandion haliaetus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-08-01 - *-12-01
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Osprey, Fall migration, expanded date range</span>

In [ ]:
file.desc <- "osprey_compl_exp_20260802"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Osprey"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-07-01", "*-12-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_osprey_compl_exp_20260802.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_osprey_compl_exp_20260802.txt 

Filters 
  Species: Pandion haliaetus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-07-01 - *-12-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Mississippi Kite, Fall migration, expanded date range</span>

In [ ]:
file.desc <- "mississippi_kite_compl_exp_20260802"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Mississippi Kite"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-07-01", "*-12-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_mississippi_kite_compl_20260726.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_mississippi_kite_compl_20260726.txt 

Filters 
  Species: Ictinia mississippiensis
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-07-01 - *-12-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Harrier, Fall migration</span>

In [27]:
file.desc <- "n_harrier_compl_20260801"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Northern Harrier"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-08-01", "*-12-01")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_name):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_n_harrier_compl_20260801.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_n_harrier_compl_20260801.txt 

Filters 
  Species: Circus hudsonius
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-08-01 - *-12-01
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

####
---

#### <span style="color: orange">Mississippi Kite, Fall migration, expanded date range - 1/2</span>

In [ ]:
file.desc <- "mississippi_kite_compl_exp1_20260803"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Mississippi Kite"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-07-01", "*-09-30")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Input 
  EBD: /media/user474/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user474/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/ebd_mississippi_kite_compl_exp1_20260803.txt 
  Sampling events: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/sed_mississippi_kite_compl_exp1_20260803.txt 

Filters 
  Species: Ictinia mississippiensis
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-07-01 - *-09-30
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Mississippi Kite, Fall migration, expanded date range - 2/2</span>

In [ ]:
file.desc <- "mississippi_kite_compl_exp2_20260803"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Mississippi Kite"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-10-01", "*-12-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Input 
  EBD: /media/user474/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user474/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/ebd_mississippi_kite_compl_exp_20260802.txt 
  Sampling events: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/sed_mississippi_kite_compl_exp_20260802.txt 

Filters 
  Species: Ictinia mississippiensis
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-10-01 - *-12-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Osprey, Fall migration, expanded date range - 1/2</span>

In [ ]:
file.desc <- "osprey_compl_exp1_20260803"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Osprey"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-07-01", "*-09-30")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Input 
  EBD: /media/user474/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user474/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/ebd_osprey_compl_exp1_20260803.txt 
  Sampling events: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/sed_osprey_compl_exp1_20260803.txt 

Filters 
  Species: Pandion haliaetus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-07-01 - *-09-30
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

#### <span style="color: orange">Osprey, Fall migration, expanded date range - 2/2</span>

In [ ]:
file.desc <- "osprey_compl_exp2_20260803"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_name = "Osprey"
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_name) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -15, longitude_max = -50, latitude_max = 65)) %>%
    auk_date(c("*-10-01", "*-12-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Input 
  EBD: /media/user474/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user474/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/ebd_osprey_compl_exp_20260803.txt 
  Sampling events: /media/user474/Storage_Root_INT/Education/Illinois/06-CS_416-Data_Visualization/Narrative_Vis_Project/output/auk/sed_osprey_compl_exp_20260803.txt 

Filters 
  Species: Pandion haliaetus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -50; Lat -15 - 65
  Years: all
  Date: *-10-01 - *-12-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

####
---

#### <span style="color: orange">Multiple raptors, Fall migration, expanded date range</span>

In [ ]:
file.desc <- "all_raptors_compl_exp_20260809"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

species_names = c(
    "Mississippi Kite",
    "Osprey",
    "Northern Harrier",
    "Sharp-shinned Hawk",
    "Merlin",
    "Swainson's Hawk",
    "Peregrine Falcon",
    "Broad-winged Hawk",
    "Swallow-tailed Kite"
)
# Full dataset filtering
f_in_ebd <- file.path(ebd_dir, ebd_filename)
f_in_sed <- file.path(sed_dir, sed_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species(species_names) %>%
    auk_bbox(bbox = c(longitude_min = -125, latitude_min = -22, longitude_max = -40, latitude_max = 60)) %>%
    auk_date(c("*-07-01", "*-12-31")) %>%
    auk_complete()
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only,
           keep = c("SCIENTIFIC NAME", "OBSERVATION COUNT",
                    "LATITUDE", "LONGITUDE", "COUNTRY", "COUNTRY CODE", "STATE", "STATE CODE",
                    "OBSERVATION DATE", "TIME OBSERVATIONS STARTED", "DURATION MINUTES",
                    "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "PROTOCOL NAME", "ALL SPECIES REPORTED",
                    "SAMPLING EVENT IDENTIFIER", "OBSERVER ID", "GROUP IDENTIFIER"),
           overwrite = TRUE)

Warning message in auk_species.auk_ebd(., species_names):
“Based on the EBD filename, it appears you should use taxonomy_version = 2025”


Input 
  EBD: /media/user797/exHDD-02/ebd_relMar-2026/ebd_relMar-2026.txt 
  Sampling events: /media/user797/exHDD-02/ebd_sampling_relMar-2026/ebd_sampling_relMar-2026.txt 

Output 
  EBD: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/ebd_all_raptors_compl_exp_20260809.txt 
  Sampling events: /home/user797/Storage_Root/Tech/Projects/CS_416_Final_Project/output/auk/sed_all_raptors_compl_exp_20260809.txt 

Filters 
  Species: Accipiter striatus, Buteo platypterus, Buteo swainsoni, Circus hudsonius, Elanoides forficatus, Falco columbarius, Falco peregrinus, Ictinia mississippiensis, Pandion haliaetus
  Countries: all
  States: all
  Counties: all
  BCRs: all
  Bounding box: Lon -125 - -40; Lat -22 - 60
  Years: all
  Date: *-07-01 - *-12-31
  Start time: all
  Last edited date: all
  Protocol: all
  Project code: all
  Duration: all
  Distance travelled: all
  Records with breeding codes only: no
  Exotic Codes: all
  Complete checklists only: yes

In [ ]:
species_names = c(
    "Mississippi Kite",
    "Osprey",
    "Northern Harrier",
    "Sharp-shinned Hawk",
    "Merlin",
    "Swainson's Hawk",
    "Peregrine Falcon",
    "Broad-winged Hawk",
    "Swallow-tailed Kite"
)
file.desc <- "all_raptors_compl_exp_20260809"
f_full_ebd <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
prefix <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, "_"))
species_files <- auk_split(f_full_ebd, species = species_names, prefix = prefix)